### Loading of Dataset

In [78]:
from datasets import load_dataset

In [79]:
df = load_dataset("rotten_tomatoes")

In [80]:
df

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [81]:
len(df)

3

In [82]:
for i in df:
    print(i,"END")

train END
validation END
test END


### Data Exploration

In [83]:
df["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 8530
})

In [84]:
df["validation"]

Dataset({
    features: ['text', 'label'],
    num_rows: 1066
})

In [85]:
df["test"]

Dataset({
    features: ['text', 'label'],
    num_rows: 1066
})

In [86]:
df["train"][0]

{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
 'label': 1}

In [87]:
df_train = df["train"].to_pandas()

In [88]:
df_train.head()

,text,label
0,the rock is destined to be the 21st century's ...,1
1,"the gorgeously elaborate continuation of "" the...",1
2,effective but too-tepid biopic,1
3,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",1


In [89]:
df_train['label'].value_counts()

label
1    4265
0    4265
Name: count, dtype: int64

### Embedding+Model Based Classification

In [90]:
from sentence_transformers import SentenceTransformer

In [91]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [92]:
train_embeddings = model.encode(df["train"]["text"],
show_progress_bar=True)
test_embeddings = model.encode(df["test"]["text"],
show_progress_bar=True)

Batches: 100%|██████████| 34/34 [00:07<00:00,  4.26it/s]


In [93]:
test_embeddings

array([[ 0.0406055 , -0.0514147 ,  0.01316864, ..., -0.00453869,
        -0.01998021,  0.00565694],
       [-0.00810177, -0.0793298 ,  0.01471894, ...,  0.08819892,
         0.05049464, -0.04774851],
       [-0.03100342, -0.02055876, -0.01974489, ..., -0.03322125,
        -0.04190544, -0.06881368],
       ...,
       [-0.08890504,  0.00279535, -0.01621611, ..., -0.05091861,
         0.01196268,  0.01434455],
       [-0.06604492,  0.01483344,  0.00404038, ...,  0.03213321,
         0.07472632,  0.15729803],
       [-0.10201585, -0.06255165,  0.06287613, ..., -0.06780174,
         0.08138094,  0.02973129]], shape=(1066, 384), dtype=float32)

### Modelling

In [94]:
from sklearn.linear_model import LogisticRegression
# Train a logistic regression on our train embeddings
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, df["train"]["label"])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [97]:
clf

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [103]:
import joblib

joblib.dump(clf, r'C:\Users\Yashita\Desktop\film_lens_ai\models\cls_logistic_v1.joblib')

['C:\\Users\\Yashita\\Desktop\\film_lens_ai\\models\\cls_logistic_v1.joblib']

### Classification

In [95]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
  performance = classification_report(
      y_true, y_pred,
      target_names=["Negative Review","Positive Review"]
  )
  print(performance)

In [96]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(df["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.78      0.77      0.77       533
Positive Review       0.77      0.78      0.77       533

       accuracy                           0.77      1066
      macro avg       0.77      0.77      0.77      1066
   weighted avg       0.77      0.77      0.77      1066



In [113]:
text = "The movie is greatest !"
a = model.encode([text])
# a = a.reshape(1, -1)

In [117]:
import time 
start_time = time.perf_counter()

In [118]:
result = clf.predict(a)

In [119]:
end_time = time.perf_counter()
total_time = end_time-start_time

In [121]:
type(total_time)

float

In [115]:
print(result)

[1]


#### Modular functions

In [76]:
s = "ABC"
print(s.lower())

abc


In [77]:
def clean_text(text):
    text_lower = text.lower()
    return text_lower

### BERT based Classification

In [ ]:
# from transformers import pipeline

# model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# pipe = pipeline(
#     model=model_path,
#     tokenizer=model_path,
#     return_all_scores=True,
#     device_map="auto"
# )

In [ ]:
# import numpy as np
# from tqdm import tqdm
# from transformers.pipelines.pt_utils import KeyDataset

# y_pred=[]
# for text in tqdm(df["test"]["text"], total=len(df["test"])):
#   output = pipe(text)
#   negative_score = output[0][0]["score"]
#   positive_score = output[0][2]["score"]
#   assignment = np.argmax([negative_score, positive_score])
#   y_pred.append(assignment)

In [ ]:
# evaluate_performance(df["test"]["label"], y_pred)